# IMPORTS

In [2]:
import duckdb
from pathlib import Path
import pandas as pd

In [3]:
path = Path.cwd().parent / "data"

# YELLOW TAXI

In [4]:
file_path = path / "yellow_taxi_unified.parquet"
out_path = path / "yellow_taxi_no_outliers.parquet"
columns = ["trip_distance", "fare_amount", "total_amount", "tip_amount", "extra", "tolls_amount", "improvement_surcharge", "congestion_surcharge", "airport_fee", "cbd_congestion_fee"]

con = duckdb.connect()
con.execute("SET memory_limit='12GB'")

In [10]:
filas_iniciales = con.execute(f"SELECT COUNT(*) FROM '{file_path}'").fetchone()[0]

In [16]:
select_parts = []
for col in columns:
    select_parts.append(f"approx_quantile({col}, 0.25) as {col}_q1")
    select_parts.append(f"approx_quantile({col}, 0.75) as {col}_q3")

query_stats = f"SELECT {', '.join(select_parts)} FROM '{file_path}'"

stats_result = con.execute(query_stats).fetchone()

In [17]:
# Mapeamos los resultados a un diccionario fácil de usar
# stats_result tiene el orden: [colA_q1, colA_q3, colB_q1, colB_q3...]
stats_map = {}
idx = 0
for col in columns:
    q1 = stats_result[idx]
    q3 = stats_result[idx+1]
    stats_map[col] = {'q1': q1, 'q3': q3}
    idx += 2

Comprobamos valores

In [18]:
print(f"{'Columna':<20} | {'Q1':<10} | {'Q3':<10} | {'IQR':<10} | {'Límite Sup (1.5)':<15}")
print("-" * 75)

for col in columns:
    s = stats_map[col]
    q1, q3 = s['q1'], s['q3']
    iqr = q3 - q1
    upper_limit = q3 + 1.5 * iqr
    
    print(f"{col:<20} | {q1:<10.2f} | {q3:<10.2f} | {iqr:<10.2f} | {upper_limit:<15.2f}")

Columna              | Q1         | Q3         | IQR        | Límite Sup (1.5)
---------------------------------------------------------------------------
trip_distance        | 1.06       | 3.47       | 2.41       | 7.09           
fare_amount          | 7.96       | 20.03      | 12.07      | 38.13          
total_amount         | 13.97      | 27.86      | 13.89      | 48.69          
tip_amount           | 0.00       | 3.86       | 3.86       | 9.66           
extra                | 0.00       | 2.50       | 2.50       | 6.25           
tolls_amount         | 0.00       | 0.00       | 0.00       | 0.00           
improvement_surcharge | 0.30       | 1.00       | 0.70       | 2.05           
congestion_surcharge | 2.50       | 2.50       | 0.00       | 2.50           
airport_fee          | 0.00       | 0.00       | 0.00       | 0.00           
cbd_congestion_fee   | 0.00       | 0.75       | 0.75       | 1.88           


Vemos que tolls_amount, congestion_surcharge y airport_fee tienen un IQR de 0, por lo que eliminará demasiadas filas. Los eliminamos de las columnas a analizar.

In [21]:
filter = ["tolls_amount","congestion_surcharge", "airport_fee", "cbd_congestion_fee"]

for col in filter: columns.remove(col)

In [22]:
where_conditions = []

for col in columns:
    stats = stats_map[col]
    iqr = stats['q3'] - stats['q1']
    lower = stats['q1'] - 1.5 * iqr
    upper = stats['q3'] + 1.5 * iqr
    
    # Añadimos la condición SQL
    # "columna BETWEEN limite_inferior AND limite_superior"
    where_conditions.append(f"({col} >= {lower} AND {col} <= {upper})")

# Unimos todas las condiciones con AND
final_filter = " AND ".join(where_conditions)

In [23]:
copy_query = f"""
COPY (
    SELECT * FROM '{file_path}' 
    WHERE {final_filter}
) TO '{out_path}' (FORMAT 'PARQUET', CODEC 'SNAPPY');
"""

con.execute(copy_query)

In [12]:
# 1. Contar filas del archivo limpio
filas_final = con.execute(f"SELECT COUNT(*) FROM '{out_path}'").fetchone()[0]

print(f"✅ Filas en el dataset inicial: {filas_iniciales:,.0f}")
print(f"✅ Filas en el dataset limpio: {filas_final:,.0f}")

perdida = filas_iniciales - filas_final
porcentaje = (filas_final / filas_iniciales) * 100

print(f"📊 Se han mantenido el {porcentaje:.2f}% de los datos.")
print(f"🗑️ Se han eliminado {perdida:,.0f} outliers.")

✅ Filas en el dataset inicial: 194,457,948
✅ Filas en el dataset limpio: 165,782,544
📊 Se han mantenido el 85.25% de los datos.
🗑️ Se han eliminado 28,675,404 outliers.
